# 02 · ESCAPE 基准 · AMP-BERT 多标签复现(训练 + 测试)

[ESCAPE](https://github.com/BCV-Uniandes/ESCAPE)(Ojeda et al., NeurIPS 2025)是一个**多标签** AMP 分类基准:
每条肽序列要预测 **5 个二元标签** —— Antibacterial、Antifungal、Antiviral、Antiparasitic、Antimicrobial
(Non-AMP 则 5 个标签全为 0)。本 notebook 把 **AMP-BERT(微调 ProtBERT-BFD)** 适配成多标签分类器,
在 ESCAPE 上复现其结果(论文报告 AMP-BERT:F1 64.7 / mAP 66.9)。

**与 notebook 01(二分类)的关键区别:**
- 模型输出 5 个 logit,损失用 `BCEWithLogitsLoss`(`problem_type='multi_label_classification'`)。
- 指标按 ESCAPE 官方 `compute_metrics` 复刻:每类 **AP**(`average_precision_score`)取平均得 **mAP**;
  每类 **F1** 取 PR 曲线上的**最大 F1**(最优阈值),再平均得整体 F1。

**训练设置(按你的要求简化):** 忽略多 seed,固定 `SEED=0`;在 **Fold1 + Fold2 合并**的训练集上训练单个模型,
在 **Test** 集上评估(不做官方的双折 ensemble)。

> 运行前请先设置:**代码执行程序 → 更改运行时类型 → GPU**。

## 0 · 环境准备
安装一个仍带经典 BERT 分词器的 `transformers` 版本(Colab 自带的太新,无法正确加载 ProtBERT-BFD),并检查 GPU。

In [ ]:
%pip install -q transformers==4.46.3 "tokenizers>=0.20,<0.21" sentencepiece

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用设备:', device)   # 期望输出 cuda

## 导入依赖与常量

In [ ]:
import os, shutil, subprocess, urllib.request
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import average_precision_score, precision_recall_curve

In [ ]:
MODEL_NAME    = 'Rostlab/prot_bert_bfd'   # 预训练模型(分词器与模型都用它)
MAX_LEN       = 200                        # 序列截断/补齐长度(与 ESCAPE 一致)
LABEL_COLUMNS = ['Antibacterial', 'Antifungal', 'Antiviral', 'Antiparasitic', 'Antimicrobial']
NUM_LABELS    = len(LABEL_COLUMNS)         # 5
SEED          = 0

## 1 · 挂载 Google Drive(保存/加载模型与重构后的数据)
模型和重构后的数据都缓存到 Drive,断线不丢;以后新开会话挂载 Drive 即可跳过重构与训练。

> 不用 Drive 的话,把 `MODEL_DIR` / `CACHE_DIR` 改成本地路径并跳过挂载(但断线会丢失)。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = '/content/drive/MyDrive/amp_bert_escape'   # 模型目录
CACHE_DIR = '/content/drive/MyDrive/escape_data'       # 重构后数据的缓存目录
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print('模型目录:', MODEL_DIR)
print('数据缓存:', CACHE_DIR)

## 2 · 准备 ESCAPE 数据(下载 + 重构缺失序列)

ESCAPE 整合了 27 个数据库,其中 **DFBP、dbAMP 3.0、DBAASP** 三个源的 License 不允许直接转发序列,
因此 Harvard Dataverse([doi:10.7910/DVN/C69MCD](https://doi.org/10.7910/DVN/C69MCD))上这些行的 `Sequence` 被**置空**,
只保留 `Hash`(= `sha256(序列)[:16]`)。官方提供 `ESCAPE_API_call.py`:从这三个数据库重新抓取序列、按 `Hash`
把空缺补回,生成完整的 `*_reconstructed.csv`。

下面的 cell 会:① 若 Drive 已有重构数据则直接用;② 否则下载原始分区 + 官方脚本并执行重构,再缓存到 Drive。

> ⚠️ 重构会访问外部数据库(含下载 dbAMP 压缩包、调用 DBAASP API),**可能耗时数分钟且依赖外部站点**。
> 若报错(某个数据库暂时不可达),重跑此 cell 即可;或在本地跑通 `ESCAPE_API_call.py` 后,把三个
> `*_reconstructed.csv` 上传到 Drive 的 `escape_data/` 目录。

In [ ]:
REC_FILES = ['Fold1_reconstructed.csv', 'Fold2_reconstructed.csv', 'Test_reconstructed.csv']

def _download(url, path):
    # Dataverse 会拒绝默认的 Python UA(403),所以伪装成浏览器
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as r, open(path, 'wb') as out:
        out.write(r.read())

if all(os.path.exists(os.path.join(CACHE_DIR, f)) for f in REC_FILES):
    print('Drive 已有重构数据,跳过重构。')
else:
    # 1) 安装重构脚本所需依赖
    subprocess.run(['pip', 'install', '-q', 'biopython'], check=True)
    # 2) 下载带缺失序列的原始分区(Dataverse,?format=original 返回 CSV)
    FILES = {'Fold1.csv': 11466752, 'Fold2.csv': 11467604, 'Test.csv': 11466751}
    for name, fid in FILES.items():
        if not os.path.exists(name):
            _download(f'https://dataverse.harvard.edu/api/access/datafile/{fid}?format=original', name)
    # 3) 下载并运行官方重构脚本(读取 cwd 的 Fold*.csv/Test.csv,生成 *_reconstructed.csv)
    _download('https://dataverse.harvard.edu/api/access/datafile/11466485?format=original',
              'ESCAPE_API_call.py')
    print('开始重构(访问外部数据库,请耐心等待)...')
    subprocess.run(['python', 'ESCAPE_API_call.py'], check=True)
    # 4) 缓存到 Drive,供以后复用
    for f in REC_FILES:
        shutil.copy(f, os.path.join(CACHE_DIR, f))
    print('重构完成,已缓存到', CACHE_DIR)

# Part A · 在 ESCAPE 上训练 AMP-BERT
若已训练并保存过模型,可跳过 Part A,直接到 Part B 加载测试。

## 3 · 加载训练数据(Fold1 + Fold2)
用重构后的两折合并成训练集并打乱。`assert` 确认没有残留的缺失序列。

In [ ]:
fold1 = pd.read_csv(os.path.join(CACHE_DIR, 'Fold1_reconstructed.csv'))
fold2 = pd.read_csv(os.path.join(CACHE_DIR, 'Fold2_reconstructed.csv'))
train_df = pd.concat([fold1, fold2], ignore_index=True).sample(frac=1, random_state=SEED)
assert train_df['Sequence'].isna().sum() == 0, '仍有缺失序列,请检查重构是否完整'
print('训练样本数:', len(train_df))
print('各标签正例数:'); print(train_df[LABEL_COLUMNS].sum())
train_df.head()

## 4 · 把序列编码成模型输入(多标签)
与二分类的唯一区别:`labels` 是长度为 5 的 **float 向量**(多热),供 `BCEWithLogitsLoss` 使用。

In [ ]:
class EscapeDataset(Dataset):
    """把 DataFrame(Sequence + 5 个标签列)编码成多标签模型输入。"""
    def __init__(self, df, tokenizer, max_len=MAX_LEN, label_columns=LABEL_COLUMNS):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.seqs = list(self.df['Sequence'])
        self.labels = self.df[label_columns].values.astype('float32')   # (N, 5)

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        seq = ' '.join(''.join(str(self.seqs[idx]).split()))   # ProtBERT 需要氨基酸间空格
        ids = self.tokenizer(seq, truncation=True, padding='max_length', max_length=self.max_len)
        sample = {k: torch.tensor(v) for k, v in ids.items()}
        sample['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)   # 5 维 float
        return sample

In [ ]:
# 加载分词器(use_fast=False 直接用 vocab.txt 的 WordPiece 分词器,避开新版本兼容问题)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False, use_fast=False)

In [ ]:
train_dataset = EscapeDataset(train_df, tokenizer)
print('train_dataset 大小:', len(train_dataset))

## 5 · 定义评估指标(复刻 ESCAPE 官方 `compute_metrics`)
`escape_scores` 严格对应 ESCAPE 仓库 `src/test_ESCAPE.py`:每类 AP + 每类 PR 曲线最大 F1,再各自取平均。
`compute_metrics` 是给 HuggingFace `Trainer` 用的包装(对 logits 取 sigmoid 得概率)。

In [ ]:
def escape_scores(y_true, y_probs):
    """返回 (每类AP, 每类F1, mAP, 整体F1)。"""
    aps, f1s = [], []
    for i in range(y_true.shape[1]):
        aps.append(average_precision_score(y_true[:, i], y_probs[:, i]))
        p, r, _ = precision_recall_curve(y_true[:, i], y_probs[:, i])
        f1s.append(np.max(2 * p * r / (p + r + 1e-8)))   # PR 曲线上的最大 F1
    return aps, f1s, float(np.mean(aps)), float(np.mean(f1s))


def compute_metrics(pred):
    y_true = pred.label_ids
    y_probs = 1.0 / (1.0 + np.exp(-pred.predictions))   # sigmoid
    _, _, mAP, macro_f1 = escape_scores(y_true, y_probs)
    return {'mAP': mAP, 'macro_f1': macro_f1}

## 6 · 定义模型(多标签头)
`num_labels=5` + `problem_type='multi_label_classification'` → 自动用 `BCEWithLogitsLoss`。

In [ ]:
def model_init():   # 必须零参数(Trainer 会把 trial 传给带参数的版本)
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, problem_type='multi_label_classification')

## 7 · 配置训练参数
ESCAPE 数据集比 Veltri 大得多(两折合计数万条),所以这里用更大的 batch、更少的 epoch。
有效批大小 = `per_device_train_batch_size` × `gradient_accumulation_steps` = 4 × 16 = 64(与 AMP-BERT 一致)。

> ProtBERT-BFD 较大,若显存不足(OOM)请调小 `per_device_train_batch_size`(如 2 或 1)并相应增大 `gradient_accumulation_steps`。
> 想先快速跑通,把 `num_train_epochs` 改成 1。

In [ ]:
OUTPUT_DIR = 'results/escape_train'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,              # 完整训练用 5(数据量大);冒烟测试用 1
    learning_rate=5e-5,
    per_device_train_batch_size=4,   # OOM 就调小
    gradient_accumulation_steps=16,  # 有效批大小 = 4 × 16 = 64
    weight_decay=0.1,
    warmup_steps=0,
    logging_steps=100,
    eval_strategy='no',
    save_strategy='no',
    fp16=True,
    seed=SEED,
    report_to='none',
)

## 8 · 开始训练

In [ ]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

## 9 · 保存模型到 Google Drive

In [ ]:
trainer.save_model(MODEL_DIR)
print('模型已保存到', MODEL_DIR)

# Part B · 在 ESCAPE 测试集上评估
从 Drive 加载模型并在 Test 集上算 mAP / F1。**新开会话直接从这里开始**时,
请先运行:环境准备(0)、导入依赖与常量、挂载 Drive(1)、准备数据(2),以及 Part A 的第 4、5 步(定义数据集类、分词器、指标函数)。

## 10 · 从 Google Drive 加载模型

In [ ]:
assert os.path.exists(os.path.join(MODEL_DIR, 'config.json')), \
    f'{MODEL_DIR} 下没有模型,请先完成 Part A 的训练与保存。'
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
print('已从 Drive 加载模型:', MODEL_DIR)

## 11 · 加载测试数据(重构后的 Test)

In [ ]:
test_df = pd.read_csv(os.path.join(CACHE_DIR, 'Test_reconstructed.csv'))
assert test_df['Sequence'].isna().sum() == 0, '测试集仍有缺失序列'
print('测试样本数:', len(test_df))
test_dataset = EscapeDataset(test_df, tokenizer)

## 12 · 评估(mAP 与 F1)
对测试集预测 → sigmoid 得概率 → 按 ESCAPE 标准计算每类 AP/F1 与整体 mAP/F1。

In [ ]:
eval_args = TrainingArguments(output_dir='results/escape_test',
                              per_device_eval_batch_size=16, report_to='none', fp16=True)
evaluator = Trainer(model=model, args=eval_args)
pred = evaluator.predict(test_dataset)

y_true  = pred.label_ids
y_probs = 1.0 / (1.0 + np.exp(-pred.predictions))   # sigmoid
aps, f1s, mAP, macro_f1 = escape_scores(y_true, y_probs)

print('每类结果:')
for name, ap, f1 in zip(LABEL_COLUMNS, aps, f1s):
    print(f'  {name:<14s}  AP={ap*100:5.1f}%   F1={f1*100:5.1f}%')
print(f'\nmAP = {mAP*100:.1f}%   |   整体 F1 = {macro_f1*100:.1f}%')
print('(论文报告 AMP-BERT:F1 64.7 / mAP 66.9)')

## 13 · 保存逐条预测概率

In [ ]:
PRED_CSV = 'results/escape_test_predictions.csv'
os.makedirs('results', exist_ok=True)
out = test_df.copy()
for i, name in enumerate(LABEL_COLUMNS):
    out[f'prob_{name}'] = y_probs[:, i]
out.to_csv(PRED_CSV, index=False)
print('已写入', PRED_CSV)